# 02 · Building the matching dataset

Each real recipient request is paired with every cornea genuinely available on its decision date. The cornea actually allocated is labelled 1. Code: `data_prep/build_matching_dataset.py`.

> **Privacy:** this notebook shows only counts, metrics, charts and study codes (R0001, Q0001), never NEBSL records. **Before committing, clear all outputs** (VS Code: `...` → *Clear All Outputs* → save).

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make `recommendation` importable whether this runs from notebooks/, recommendation/ or ml/
ML = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "recommendation" / "config.py").exists())
sys.path.insert(0, str(ML))

import pandas as pd
from IPython.display import Image, display
from recommendation.config import load_config

pd.set_option("display.width", 160)
cfg = load_config()
DATASET = cfg["paths"]["matching_dataset"]
OUTPUTS = Path(cfg["paths"]["outputs"])
print("Dataset:", Path(DATASET).name)

## 1. How the dataset was built (build notes)

In [ ]:
pd.read_excel(DATASET, sheet_name="Build_Notes").set_index("Item")

## 2. Size and balance

In [ ]:
pairs = pd.read_excel(DATASET, sheet_name="Ranking_Pairs")
pool = pairs.groupby("Request_ID").size()
pd.Series({"requests": pool.size, "ranking pairs": len(pairs),
           "candidates per request (mean)": round(pool.mean(), 1),
           "candidates per request (median)": pool.median(),
           "positive rate": f"{pairs['Label_Allocated'].mean():.1%}"}).to_frame("value")

In [ ]:
display(Image(filename=str(OUTPUTS / "eda" / "01_candidates_per_request.png")))

## 3. Correctness checks (0 = pass)

These prove the ranking task is built without leakage: every candidate was really ready, in date and not already given to someone else.

In [ ]:
from recommendation.data_prep.eda import validate_matching

checks = pd.Series(validate_matching(DATASET)).rename("problems").to_frame()
checks["result"] = checks["problems"].map(lambda v: "PASS" if v == 0 else "CHECK")
checks

## 4. What past allocations favour

In [ ]:
display(Image(filename=str(OUTPUTS / "eda" / "05_grade_gap_by_label.png")))
chosen = pairs["Label_Allocated"] == 1
pd.DataFrame({
    "chosen": [(pairs.loc[chosen, "Grade_Gap"] == 0).mean(), pairs.loc[chosen, "Cand_Days_In_Storage"].mean(),
               (pairs.loc[chosen, "Cand_Serology_Clear"] == "Yes").mean()],
    "not chosen": [(pairs.loc[~chosen, "Grade_Gap"] == 0).mean(), pairs.loc[~chosen, "Cand_Days_In_Storage"].mean(),
                   (pairs.loc[~chosen, "Cand_Serology_Clear"] == "Yes").mean()]},
    index=["exact grade match (share)", "days in storage (mean)", "serology confirmed (share)"]).round(2)